# 🦻 MEM-EAR — Modified Version
### Changes from original:
- ✅ `EMBED_DIM` : 512 → **256**
- ✅ `EPOCHS` : 30 → **5**
- ✅ `BATCH_SIZE` : 32 → **32** (same)
- ✅ `EarNormalizer` / `normalize_ear` / `cv2` **hataaya** (dataset already aligned hai)
- ✅ Baaki sab (iResNet100, EfficientNet-B3, ConvNeXt-tiny, ArcFace, Ensemble) same rakha

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────
import os, math, time, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

In [ ]:
# ── CELL 2: Config ────────────────────────────────────────────
DATA_ROOT  = Path('/kaggle/input/datasets/madhvii0911/uerc-oriented22/UERC26_Oriented/data/public')
OUTPUT_DIR = Path('/kaggle/working/ear_recognition')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE   = 112
EMBED_DIM  = 256       # ← 512 se 256 (half)
BATCH_SIZE = 32
EPOCHS     = 5         # ← 30 se 5
LR         = 1e-4
ARCFACE_S  = 64.0
ARCFACE_M  = 0.5
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'[INFO] Device   : {DEVICE}')
print(f'[INFO] EMBED_DIM: {EMBED_DIM}')
print(f'[INFO] EPOCHS   : {EPOCHS}')

In [ ]:
# ── CELL 3: Dataset Discovery ─────────────────────────────────
def discover_dataset(root: Path):
    records = []
    VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    with os.scandir(root) as persons:
        for person in persons:
            if not person.is_dir():
                continue
            with os.scandir(person.path) as images:
                for img in images:
                    if img.is_file() and Path(img.name).suffix.lower() in VALID_EXT:
                        records.append({'path': img.path, 'person_id': person.name})
    df = pd.DataFrame(records)
    le = LabelEncoder()
    df['label']  = le.fit_transform(df['person_id'])
    n_classes    = df['label'].nunique()
    print(f'[DATA] {len(df)} images | {n_classes} identities')
    return df, le, n_classes

df, label_enc, n_classes = discover_dataset(DATA_ROOT)
df.head()

In [ ]:
# ── CELL 4: Class Weights ─────────────────────────────────────
def compute_class_weights(df: pd.DataFrame) -> torch.Tensor:
    counts    = Counter(df['label'])
    n_classes = len(counts)
    total     = len(df)
    weights   = torch.zeros(n_classes, dtype=torch.float32)
    for cls, cnt in counts.items():
        weights[cls] = total / (n_classes * cnt)
    weights = weights / weights.mean()
    return weights

class_weights = compute_class_weights(df)
print(f'Class weights shape: {class_weights.shape}')

In [ ]:
# ── CELL 5: Transforms (NO EarNormalizer — dataset already aligned) ──
# cv2 / CLAHE hata diya, seedha resize + augment

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print('Transforms ready (no EarNormalizer)')

In [ ]:
# ── CELL 6: EarDataset ────────────────────────────────────────
class EarDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['path']).convert('RGB')
        label = int(row['label'])
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# ── CELL 7: DataLoaders ───────────────────────────────────────
def build_loaders(df, val_frac=0.15):
    train_df, val_df = train_test_split(
        df, test_size=val_frac, stratify=df['label'], random_state=SEED)

    train_ds = EarDataset(train_df, train_transform)
    val_ds   = EarDataset(val_df,   eval_transform)

    class_counts   = Counter(train_df['label'].values)
    sample_weights = [1.0 / class_counts[l] for l in train_df['label'].values]
    sampler        = WeightedRandomSampler(sample_weights,
                                           num_samples=len(sample_weights),
                                           replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)
    print(f'[LOADERS] Train={len(train_df)} | Val={len(val_df)}')
    return train_loader, val_loader, train_df, val_df

train_loader, val_loader, train_df, val_df = build_loaders(df)

In [ ]:
# ── CELL 8: ConvNeXt-tiny Embedder ────────────────────────────
class ConvNeXtTinyEmbedder(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM, pretrained: bool = True):
        super().__init__()
        self.backbone  = timm.create_model('convnext_tiny',
                                            pretrained=pretrained,
                                            num_classes=0)
        feat_dim       = self.backbone.num_features
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, embed_dim),
            nn.BatchNorm1d(embed_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb  = self.projector(feat)
        return F.normalize(emb, dim=1)

In [ ]:
# ── CELL 9: EfficientNet-B3 Embedder ──────────────────────────
class EfficientNetB3Embedder(nn.Module):
    def __init__(self, embed_dim: int = EMBED_DIM, pretrained: bool = True):
        super().__init__()
        self.backbone  = timm.create_model('efficientnet_b3',
                                            pretrained=pretrained,
                                            num_classes=0)
        feat_dim       = self.backbone.num_features
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, embed_dim),
            nn.BatchNorm1d(embed_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb  = self.projector(feat)
        return F.normalize(emb, dim=1)

In [ ]:
# ── CELL 10: iResNet-100 ──────────────────────────────────────
class IBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bn1       = nn.BatchNorm2d(inplanes)
        self.conv1     = nn.Conv2d(inplanes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2       = nn.BatchNorm2d(planes)
        self.prelu     = nn.PReLU(planes)
        self.conv2     = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn3       = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride    = stride

    def forward(self, x):
        residual = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return out


class IResNet(nn.Module):
    def __init__(self, block, layers, embed_dim=EMBED_DIM, dropout=0.0):
        super().__init__()
        self.inplanes = 64
        self.conv1    = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(64)
        self.prelu    = nn.PReLU(64)
        self.layer1   = self._make_layer(block, 64,  layers[0], stride=2)
        self.layer2   = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3   = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4   = self._make_layer(block, 512, layers[3], stride=2)
        self.bn2      = nn.BatchNorm2d(512)
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(512 * 7 * 7, embed_dim)
        self.features = nn.BatchNorm1d(embed_dim)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion)
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x);  x = self.bn1(x);  x = self.prelu(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        x = self.bn2(x)
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.features(x)
        return F.normalize(x, dim=1)


def iresnet100(embed_dim=EMBED_DIM):
    return IResNet(IBasicBlock, [3, 13, 30, 3], embed_dim=embed_dim)

In [ ]:
# ── CELL 11: Weighted ArcFace Loss ────────────────────────────
class WeightedArcFaceLoss(nn.Module):
    def __init__(self, embed_dim: int, n_classes: int,
                 class_weights: torch.Tensor,
                 s: float = ARCFACE_S, m: float = ARCFACE_M):
        super().__init__()
        self.s      = s
        self.m      = m
        self.weight = nn.Parameter(torch.FloatTensor(n_classes, embed_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m  = math.cos(m)
        self.sin_m  = math.sin(m)
        self.th     = math.cos(math.pi - m)
        self.mm     = math.sin(math.pi - m) * m
        self.register_buffer('class_weights', class_weights)

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor):
        cosine  = F.linear(embeddings, F.normalize(self.weight))
        sine    = torch.sqrt(1.0 - cosine.pow(2).clamp(0, 1))
        phi     = cosine * self.cos_m - sine * self.sin_m
        phi     = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output  = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output  = output * self.s
        w       = self.class_weights[labels]
        loss    = F.cross_entropy(output, labels, reduction='none')
        loss    = (loss * w).mean()
        return loss

In [ ]:
# ── CELL 12: Train One Epoch ──────────────────────────────────
def train_one_epoch(model, criterion, optimizer, loader, device):
    model.train(); criterion.train()
    total_loss, total = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        emb  = model(imgs)
        loss = criterion(emb, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(criterion.parameters()), 5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total      += len(labels)
    return total_loss / total

In [ ]:
# ── CELL 13: Evaluate (Rank-1) ────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    embeddings, label_list = [], []
    for imgs, labels in loader:
        emb = model(imgs.to(device))
        embeddings.append(emb.cpu())
        label_list.extend(labels.tolist())
    embeddings = torch.cat(embeddings, dim=0).numpy()
    label_arr  = np.array(label_list)

    seen, gallery_emb, gallery_lbl = set(), [], []
    probe_emb,  probe_lbl          = [], []
    for i, lbl in enumerate(label_arr):
        if lbl not in seen:
            gallery_emb.append(embeddings[i]); gallery_lbl.append(lbl); seen.add(lbl)
        else:
            probe_emb.append(embeddings[i]);   probe_lbl.append(lbl)

    if not probe_emb:
        return 0.0

    G        = np.array(gallery_emb)
    P        = np.array(probe_emb)
    sims     = P @ G.T
    pred_idx = sims.argmax(axis=1)
    predicted = np.array(gallery_lbl)[pred_idx]
    rank1     = (predicted == np.array(probe_lbl)).mean()
    return rank1

In [ ]:
# ── CELL 14: Train Model (5 Epochs) ──────────────────────────
def train_model(model_name: str, model: nn.Module,
                train_loader, val_loader,
                n_classes: int, class_weights: torch.Tensor):
    model     = model.to(DEVICE)
    criterion = WeightedArcFaceLoss(
        EMBED_DIM, n_classes, class_weights.to(DEVICE)).to(DEVICE)
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(criterion.parameters()),
        lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6)

    best_rank1, best_state = 0.0, None
    history = []

    for epoch in range(1, EPOCHS + 1):
        t0    = time.time()
        loss  = train_one_epoch(model, criterion, optimizer, train_loader, DEVICE)
        rank1 = evaluate(model, val_loader, DEVICE)
        scheduler.step()
        elapsed = time.time() - t0
        history.append({'epoch': epoch, 'loss': loss, 'rank1': rank1})
        print(f'  [{model_name}] E{epoch:02d}/{EPOCHS} '
              f'loss={loss:.4f}  Rank-1={rank1*100:.2f}%  ({elapsed:.1f}s)')

        if rank1 > best_rank1:
            best_rank1 = rank1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), OUTPUT_DIR / f'{model_name}_best.pt')
    print(f'  [{model_name}] Best Rank-1 = {best_rank1*100:.2f}%\n')
    return model, best_rank1, history

In [ ]:
# ── CELL 15: Extract Embeddings ───────────────────────────────
@torch.no_grad()
def extract_embeddings(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    embs, labels = [], []
    for imgs, lbls in loader:
        embs.append(model(imgs.to(device)).cpu())
        labels.extend(lbls.tolist())
    return torch.cat(embs, 0).numpy(), np.array(labels)


def ensemble_embeddings(models_weights: list, loaders: tuple):
    train_loader, val_loader = loaders
    total_weight = sum(w for _, w in models_weights)
    combined_train_emb = combined_val_emb = None
    train_labels_ref   = val_labels_ref   = None

    for model, weight in models_weights:
        w  = weight / total_weight
        te, tl = extract_embeddings(model, train_loader, DEVICE)
        ve, vl = extract_embeddings(model, val_loader,   DEVICE)
        if combined_train_emb is None:
            combined_train_emb = w * te
            combined_val_emb   = w * ve
            train_labels_ref   = tl
            val_labels_ref     = vl
        else:
            combined_train_emb += w * te
            combined_val_emb   += w * ve

    def l2(x):
        n = np.linalg.norm(x, axis=1, keepdims=True)
        return x / (n + 1e-8)

    return (l2(combined_train_emb), train_labels_ref,
            l2(combined_val_emb),   val_labels_ref)

In [ ]:
# ── CELL 16: Ensemble Evaluate ────────────────────────────────
def ensemble_evaluate(val_emb: np.ndarray, val_labels: np.ndarray):
    seen, gallery_e, gallery_l = set(), [], []
    probe_e, probe_l           = [], []
    for i, lbl in enumerate(val_labels):
        if lbl not in seen:
            gallery_e.append(val_emb[i]); gallery_l.append(lbl); seen.add(lbl)
        else:
            probe_e.append(val_emb[i]);   probe_l.append(lbl)

    if not probe_e:
        print('[WARN] No probe samples — skipping eval')
        return {}

    G, GL = np.array(gallery_e), np.array(gallery_l)
    P, PL = np.array(probe_e),   np.array(probe_l)
    sims  = P @ G.T

    rank1    = (GL[sims.argmax(1)] == PL).mean()
    top5_idx = np.argsort(-sims, axis=1)[:, :5]
    rank5    = np.array([PL[i] in GL[top5_idx[i]] for i in range(len(PL))]).mean()

    # Approx EER
    same_scores, diff_scores = [], []
    for i in range(len(PL)):
        for j in range(len(GL)):
            s = sims[i, j]
            (same_scores if PL[i] == GL[j] else diff_scores).append(s)
    all_scores = np.array(same_scores + diff_scores)
    all_labels = np.array([1]*len(same_scores) + [0]*len(diff_scores))
    thresholds = np.linspace(all_scores.min(), all_scores.max(), 200)
    best_eer   = 1.0
    for t in thresholds:
        far = ((all_scores[all_labels == 0]) >= t).mean()
        frr = ((all_scores[all_labels == 1]) <  t).mean()
        if abs(far - frr) < abs(best_eer - 0.5):
            best_eer = (far + frr) / 2

    results = {'Rank-1': rank1, 'Rank-5': rank5, 'EER': best_eer}
    print('\n' + '='*50)
    print('  ENSEMBLE EVALUATION RESULTS')
    print('='*50)
    for k, v in results.items():
        print(f'  {k:10s}: {v*100:.2f}%')
    print('='*50 + '\n')
    return results

In [ ]:
# ── CELL 17: Build Backbones ──────────────────────────────────
backbones = {
    'convnext_tiny'   : ConvNeXtTinyEmbedder(EMBED_DIM),
    'efficientnet_b3' : EfficientNetB3Embedder(EMBED_DIM),
    'iresnet100'      : iresnet100(EMBED_DIM),
}

for name, m in backbones.items():
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'  {name:20s}: {n:.1f}M params  (embed={EMBED_DIM})')

In [ ]:
# ── CELL 18: Train All Models ─────────────────────────────────
trained     = {}
all_history = {}

for name, model in backbones.items():
    print(f'\n>>> Training {name.upper()} <<<<')
    m, rank1, hist = train_model(name, model,
                                  train_loader, val_loader,
                                  n_classes, class_weights)
    trained[name]     = (m, rank1)
    all_history[name] = hist

In [ ]:
# ── CELL 19: Ensemble Weights ─────────────────────────────────
print('\n>>> Ensemble Weights (proportional to Rank-1) <<<')
models_weights = []
for name, (m, rank1) in trained.items():
    print(f'  {name:25s}: rank1={rank1*100:.2f}%  weight={rank1:.4f}')
    models_weights.append((m, rank1))

In [ ]:
# ── CELL 20: Final Ensemble Evaluation ───────────────────────
_, _, fused_val_emb, fused_val_lbl = ensemble_embeddings(
    models_weights, (train_loader, val_loader))

results = ensemble_evaluate(fused_val_emb, fused_val_lbl)